In [2]:
print("Hello, world today!...")

Hello, world today!...


In [3]:
#uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-commuinity

In [4]:
#! uv pip install langchain langchain-huggingface langchain-community rapidocr-onnxruntime python-dotenv sentence-transformers

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

### Data Ingestion

from langchain.document_loaders import TextLoader

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/Agentic AI.txt", encoding="utf8")
documents = loader.load()

print(f"Successfully loaded {len(documents)} document(s).")

/home/ahmad/Documents/projects/rag_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 1 document(s).


In [7]:
documents[0].page_content[:500]      # print the first 500 characters of the first document

'Understanding Agentic AI\n\nAgentic AI: Understanding the Next Frontier of Intelligent SystemsSlide 1: Title SlideTitle: Agentic AI: Understanding Agentic AISubtitle: Moving From Passive Automation to Autonomous SubsystemsPresenter: [Your Name/Title]Context: Technology Presentation & Executive BriefingSlide 2: The Evolution of Artificial IntelligenceShifting ParadigmsArtificial Intelligence is undergoing a fundamental shift in how it interacts with human intent, transitioning from static generatio'

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

In [9]:
text_chunks = text_splitter.split_documents(documents)

In [10]:
text_chunks

[Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Agentic AI: Understanding the Next Frontier of Intelligent SystemsSlide 1: Title SlideTitle: Agentic AI: Understanding Agentic AISubtitle: Moving From Passive Automation to Autonomous'),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='to Autonomous SubsystemsPresenter: [Your Name/Title]Context: Technology Presentation & Executive BriefingSlide 2: The Evolution of Artificial IntelligenceShifting ParadigmsArtificial Intelligence is'),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Intelligence is undergoing a fundamental shift in how it interacts with human intent, transitioning from static generation to active execution.Generative AI (Past to Present): Proactive text, image,'),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='text, image, or code gener

In [11]:
! uv pip install pip faiss-cpu
print("faiss installed...")

Using Python 3.12.13 environment at: /home/ahmad/Documents/projects/rag_project/.venv
Checked 2 packages in 48ms
faiss installed...


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [13]:
# 1. LOAD THE SEARCH ENGINE (Lightweight - ~500MB RAM)
# This model converts words to math vectors. 
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1675.57it/s]


In [14]:
vector_store = FAISS.from_documents(text_chunks, embeddings)

In [15]:
vector_store

In [16]:
retriever = vector_store.as_retriever()

In [17]:
# Perform a similarity search

query = "What is the key characteristic of Agentic AI?"
docs = vector_store.similarity_search(query, k=4)

# Display the result
for i, doc in enumerate(docs):
    print(f"Documents {i+1}:")
    print(doc.page_content)
    print("_" * 50)

Documents 1:
AI system is considered agentic when it possesses degree of autonomy, situational awareness, and the ability to act upon an environment to achieve a specific outcome.Key Takeaway: Unlike traditional
__________________________________________________
Documents 2:
Understanding Agentic AI
__________________________________________________
Documents 3:
the objective; the AI autonomously determines the plan, selects the tools, handles exceptions, and executes the workflow.Slide 3: What Makes an AI "Agentic"?The Core DefinitionAn AI system is
__________________________________________________
Documents 4:
4: Architectural Blueprint of an AI AgentAn agent functions through an interconnected loop of perception, reasoning, and action.Plaintext       ┌──────────────────────────────────────┐
__________________________________________________


In [18]:
from langchain_core.prompts import ChatPromptTemplate

template = """ You are an assistant for question-answering tasks. Use the following piece of retrieved context
to answer the question. If you do not know the answer, just say that you do not know. Use ten sentences maximum
and keep the answer concise. 

Question: {question}
Context: {context}
Answer:
"""

In [19]:
prompt = ChatPromptTemplate.from_template(template)

In [20]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=' You are an assistant for question-answering tasks. Use the following piece of retrieved context\nto answer the question. If you do not know the answer, just say that you do not know. Use ten sentences maximum\nand keep the answer concise. \n\nQuestion: {question}\nContext: {context}\nAnswer:\n'), additional_kwargs={})])

In [21]:
from langchain_core.output_parsers import StrOutputParser

In [22]:
output_parser=StrOutputParser()

In [23]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Load FLAN-T5 model
pipe = pipeline("text-generation", model="google/flan-t5-base")
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2655.81it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', '

In [25]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Build RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [26]:
rag_chain.invoke("Tell me about agentic AI")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human:  You are an assistant for question-answering tasks. Use the following piece of retrieved context\nto answer the question. If you do not know the answer, just say that you do not know. Use ten sentences maximum\nand keep the answer concise. \n\nQuestion: Tell me about agentic AI\nContext: [Document(id=\'dd256394-8b80-4545-8b57-99d55122415d\', metadata={\'source\': \'../data/Agentic AI.txt\'}, page_content=\'Understanding Agentic AI\'), Document(id=\'598491b9-a986-4a83-ab60-9db9ec8c7039\', metadata={\'source\': \'../data/Agentic AI.txt\'}, page_content=\'AI system is considered agentic when it possesses degree of autonomy, situational awareness, and the ability to act upon an environment to achieve a specific outcome.Key Takeaway: Unlike traditional\'), Document(id=\'30687aa9-7c5d-47a0-82f4-74baa2f3c284\', metadata={\'source\': \'../data/Agentic AI.txt\'}, page_content=\'the objective; the AI autonomously determines the plan, selects the tools, handles exceptions, and executes th